In [ ]:
library(tidyverse)
library(readxl)
library(lubridate)

In [86]:
# =========================
# 1. CARGA DE DATOS
# =========================
df <- read_excel("indices_precios_base_2019.xlsx")

df <- df %>%
  mutate(
    fecha = as.Date(fecha),
    year  = year(fecha),
    month = month(fecha),
    ym    = year * 12 + month
  ) %>%
  arrange(ym)

In [87]:
# =========================
# 2. LOG-DIFERENCIAS (IGUAL A STATA)
# =========================
vars <- c(
  "ferrocarril","autotransporte_carga",
  "alimentos_bebidas_tabaco","ropa_calzado_accesorios",
  "vivienda","muebles_aparatos_domesticos",
  "salud_cuidado_personal","transporte",
  "educacion_esparcimiento","otros_servicios",
  "inpp_sin_pet_con_serv","inpc"
)

for(v in vars){
  df[[paste0("ln_", v)]] <- log(df[[v]])
  df[[paste0("dln_", v)]] <- df[[paste0("ln_", v)]] - lag(df[[paste0("ln_", v)]])
}

# =========================
# 3. VARIABLES CLAVE
# =========================
df <- df %>%
  mutate(
    infl_bienes = dln_alimentos_bebidas_tabaco,
    infl_bienes_amplia = (
      dln_alimentos_bebidas_tabaco +
      dln_ropa_calzado_accesorios +
      dln_muebles_aparatos_domesticos
    ) / 3,
    dln_ferro = dln_ferrocarril,
    dln_autot = dln_autotransporte_carga
  )

# =========================
# 4. DESESTACIONALIZACIÓN (CLON STATA EXACTO)
# =========================
sa <- function(x, month){
  
  dummies <- model.matrix(~ factor(month))
  
  fit <- lm(x ~ dummies[, -1], na.action = na.exclude)
  resid(fit)
}

df <- df %>%
  mutate(
    infl_bienes_sa = sa(infl_bienes, month),
    infl_bienes_amplia_sa = sa(infl_bienes_amplia, month),
    dln_ferro_sa = sa(dln_ferro, month),
    dln_autot_sa = sa(dln_autot, month),
    dln_inpc_sa = sa(dln_inpc, month),
    dln_ropa_sa = sa(dln_ropa_calzado_accesorios, month),
    dln_vivienda_sa = sa(dln_vivienda, month),
    dln_muebles_sa = sa(dln_muebles_aparatos_domesticos, month),
    dln_salud_sa = sa(dln_salud_cuidado_personal, month),
    dln_transporte_sa = sa(dln_transporte, month),
    dln_educacion_sa = sa(dln_educacion_esparcimiento, month),
    dln_otros_servicios_sa = sa(dln_otros_servicios, month),
    dln_inpp_sa = sa(dln_inpp_sin_pet_con_serv, month)
  )

# =========================
# 5. REZAGOS (IGUAL A STATA tsset)
# =========================
for(l in 0:4){
  df[[paste0("L",l,"_ferro")]] <- lag(df$dln_ferro_sa, l)
  df[[paste0("L",l,"_autot")]] <- lag(df$dln_autot_sa, l)
}


In [88]:
# =========================
# 6. FUNCIÓN ARDL (CLON STATA)
# =========================
run_ardl_manual <- function(data, y_var){
  
  df_reg <- data %>%
    filter(year > 1992 | (year == 1992 & month >= 6)) %>%
    select(
      y = all_of(y_var),
      L0_ferro, L1_ferro, L2_ferro, L3_ferro, L4_ferro,
      L0_autot, L1_autot, L2_autot, L3_autot, L4_autot
    )
  
  df_reg <- df_reg %>%
    mutate(
      L1_y = lag(y,1),
      L2_y = lag(y,2),
      L3_y = lag(y,3),
      L4_y = lag(y,4)
    ) %>%
    drop_na()
  
  fit <- lm(
    y ~ L1_y + L2_y + L3_y + L4_y +
      L0_ferro + L1_ferro + L2_ferro + L3_ferro + L4_ferro +
      L0_autot + L1_autot + L2_autot + L3_autot + L4_autot,
    data = df_reg
  )
  
  return(fit)
}


In [94]:
# =========================
# 7. TABLA 1
# =========================
modelo <- run_ardl_manual(df, "infl_bienes_sa")

cat("\n--- TABLA 1 ---\n")
print(paste("Número de observaciones:", nobs(modelo)))
print(summary(modelo))


--- TABLA 1 ---


[1] "Número de observaciones: 399"

Call:
lm(formula = y ~ L1_y + L2_y + L3_y + L4_y + L0_ferro + L1_ferro + 
    L2_ferro + L3_ferro + L4_ferro + L0_autot + L1_autot + L2_autot + 
    L3_autot + L4_autot, data = df_reg)

Residuals:
       Min         1Q     Median         3Q        Max 
-0.0277033 -0.0036766  0.0002515  0.0037382  0.0225940 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -0.0013063  0.0005239  -2.494  0.01306 *  
L1_y         0.5230868  0.0514458  10.168  < 2e-16 ***
L2_y        -0.1031466  0.0579040  -1.781  0.07565 .  
L3_y         0.0291571  0.0582927   0.500  0.61723    
L4_y         0.0673066  0.0503919   1.336  0.18245    
L0_ferro     0.0153856  0.0187629   0.820  0.41273    
L1_ferro     0.0594622  0.0189062   3.145  0.00179 ** 
L2_ferro    -0.0230084  0.0191879  -1.199  0.23122    
L3_ferro    -0.0095818  0.0191932  -0.499  0.61790    
L4_ferro     0.0092684  0.0191431   0.484  0.62854    
L0_autot     0.1849716  0.0263846  

In [90]:
# =========================
# 8. TABLA 2 (PASS-THROUGH)
# =========================
cf <- coef(modelo)

ferro_cp <- cf["L0_ferro"] * 10
ferro_lp <- sum(cf[paste0("L",0:4,"_ferro")]) * 10

autot_cp <- cf["L0_autot"] * 10
autot_lp <- sum(cf[paste0("L",0:4,"_autot")]) * 10

ratio_lp <- autot_lp / ferro_lp

cat("\n--- TABLA 2 ---\n")
cat("Ferro CP:", ferro_cp, "\n")
cat("Ferro LP:", ferro_lp, "\n")
cat("Autot CP:", autot_cp, "\n")
cat("Autot LP:", autot_lp, "\n")
cat("Ratio LP:", ratio_lp, "\n")



--- TABLA 2 ---
Ferro CP: 0.1538555 
Ferro LP: 0.5152596 
Autot CP: 1.849716 
Autot LP: 2.738331 
Ratio LP: 5.314469 


In [92]:
# =========================
# 9. TABLA 3
# =========================
specs <- c(
  "Alim/Beb/Tab" = "infl_bienes_sa",
  "INPC general" = "dln_inpc_sa",
  "Ropa y Calzado" = "dln_ropa_sa",
  "Vivienda" = "dln_vivienda_sa",
  "Muebles" = "dln_muebles_sa",
  "Salud" = "dln_salud_sa",
  "Transporte" = "dln_transporte_sa",
  "Educacion" = "dln_educacion_sa",
  "Otros_serv" = "dln_otros_servicios_sa",
  "Bienes_amplia" = "infl_bienes_amplia_sa"
)

tabla_3 <- bind_rows(
  lapply(names(specs), function(n){
    
    mod <- run_ardl_manual(df, specs[[n]])
    cf  <- coef(mod)
    
    tibble(
      Variable = n,
      Ferro_CP = cf["L0_ferro"] * 10,
      Ferro_LP = sum(cf[paste0("L",0:4,"_ferro")]) * 10,
      Autot_CP = cf["L0_autot"] * 10,
      Autot_LP = sum(cf[paste0("L",0:4,"_autot")]) * 10
    )
  })
) %>%
  mutate(across(where(is.numeric), \(x) round(x, 2)))

cat("\n--- TABLA 3 ---\n")
tabla_3


--- TABLA 3 ---


Variable,Ferro_CP,Ferro_LP,Autot_CP,Autot_LP
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Alim/Beb/Tab,0.15,0.52,1.85,2.74
INPC general,0.45,0.34,1.27,1.49
Ropa y Calzado,0.43,0.67,0.89,1.59
Vivienda,0.59,0.24,1.62,2.99
Muebles,0.37,-0.07,1.05,1.82
Salud,0.32,0.01,0.95,1.24
Transporte,1.37,2.33,2.13,3.95
Educacion,0.21,0.14,0.89,1.64
Otros_serv,0.46,0.52,1.50,1.40
